Task 1 — Exploratory Data Analysis & Visualization  
Conduct an exploratory analysis of the dataset. Produce a minimum of five visualizations covering feature
distributions, correlations, and categorical breakdowns. Each visualization must be accompanied by a written
interpretation explaining what it reveals about the data.

In [ ]:
# Libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.feature_selection import SelectKBest, f_regression
import random
import pickle
warnings.filterwarnings('ignore')

In [ ]:
#Task 1 EDA :
# Load the data
data = pd.read_csv(r'./premier_league.csv')
# Basic info of the data
print("general info of the dataset:")
print(data.info())



In [ ]:
print(data.head())
print("="*80)
print(data.tail())
print("="*80)

In [ ]:
#convert data types
data['game_id'] = data['game_id'].astype('category')
data['date'] = pd.to_datetime(data['date'])
data['year']=data['date'].dt.year


In [ ]:
duplicates = data.duplicated().sum()
print(f"Number of duplicates: {duplicates}")

In [ ]:
# check out the dimension of the dataset
data.shape

In [ ]:
for col in data.columns:
     print(f"Column {col}:")
     print(f"Number of unique values: {len(data[col].value_counts())}")
     print(f"Number of null values: {data[col].isnull().sum()}")
     print("------------------------------")

In [ ]:
#numerical_features & categorical_features
numerical_features = data.select_dtypes(include=['int64', 'float64']).columns.tolist()
categorical_features = data.select_dtypes(include=['object', 'category']).columns.tolist()
print("Numerical features:", numerical_features)
print("Caterorical features:", categorical_features)

In [ ]:
print("value count in every column for each row:")
for col in categorical_features:
     print(f"Column {col}:")
     print(data[col].value_counts())
     print("------------------------------")

In [ ]:
# Descriptive statistics
print("\nDescriptive statistics:")
print(data[numerical_features].describe())

In [ ]:
# Descriptive statistics
print("\nDescriptive Categorical:")
print(data[categorical_features].describe())

In [ ]:
for col in categorical_features:
     print(f"Column {col}:")
     print(f"unique values in each category col: {data[col].unique()}")
     print("------------------------------")

In [ ]:
#visualizations

In [ ]:
#1.outlier detction
# boxplot to detect outliers
fig = make_subplots(
    rows=len(numerical_features),
    cols=1,
    subplot_titles=[f"Boxplot of {col}" for col in numerical_features]
)

for i, col in enumerate(numerical_features, start=1):
    fig.add_trace(
        go.Box(y=data[col], name=col),
        row=i, col=1
    )

fig.update_layout(
    title="Boxplots",
    showlegend=False,
    height=300 * len(numerical_features)
)

fig.show()

as we see most of numeric cols have outliers

In [ ]:
#2.histogram for all numerical cols
fig = make_subplots(
    rows=len(numerical_features),
    cols=1,
    subplot_titles=[f"Distribution of {col}" for col in numerical_features]
)

for i, col in enumerate(numerical_features, start=1):
    fig.add_trace(
        go.Histogram(x=data[col], name=col),
        row=i, col=1
    )

fig.update_layout(
    title="Distributions",
    showlegend=False,
    height=300 * len(numerical_features)
)

fig.show()

frequency histogram to all numeric cols , another way to see the distrubition of data

In [ ]:
#3.Correlation Heatmap
corr = data.select_dtypes(include='number').corr()

fig = px.imshow(corr,
                text_auto='.2f',
                aspect='auto',
                color_continuous_scale='RdBu_r',
                title='Correlation Heatmap')
fig.update_layout(width=1000, height=800)
fig.show()

this heatmap show correlation with all numeric cols , high corr is shown in red

In [ ]:
#4.top_scorers
top_scorers = data.groupby('player_name')['goals'].sum().sort_values(ascending=False).head(10)

plt.figure(figsize=(10,6))
plt.bar(top_scorers.index, top_scorers.values, color='hotpink')

plt.title("Top 10 Scorers")
plt.xlabel("Player")
plt.ylabel("Goals")
plt.xticks(rotation=45)

plt.show()

the top player to score last 4 years is Mo Salah , then haaland comes

In [ ]:
#5.top 5 clubs vs player values
club_values = data.groupby('club_name')['market_value_in_eur'].sum().reset_index()
#top 5 clubs
top5 = club_values.sort_values('market_value_in_eur', ascending=False).head(5)

fig = px.bar(
    top5,
    x='club_name',
    y='market_value_in_eur',
    title='Top 5 Clubs by Market Value (EUR)',
    text='market_value_in_eur'
)

fig.update_layout(xaxis_title="Club", yaxis_title="Total Market Value (€)")
fig.show()

Arsenal is the top club to have high value players, then man city comes

In [ ]:
#6.line chart year by average price
# Group by year and calculate average price
price_by_year = data.groupby('year')['market_value_in_eur'].mean().reset_index()
price_by_year = price_by_year.sort_values('year')  # Ensure order

# Create the plot
plt.figure(figsize=(10, 6))
plt.plot(price_by_year['year'], price_by_year['market_value_in_eur'],
         marker='o', linewidth=2)

# Add labels and title
plt.xlabel('Year')
plt.ylabel('Average players values')
plt.title('players values Over Time')
plt.grid(True, alpha=0.3)
plt.show()

players value increases over time

In [ ]:
#7.club power vs wins
club_win = data.groupby('club_name')['is_win'].mean()
club_power = data.groupby('club_name')['Club_International_Power'].mean()

plt.figure(figsize=(8,6))

plt.scatter(club_power, club_win, color='purple')

plt.title("Club Strength vs Win Rate")
plt.xlabel("Club Power")
plt.ylabel("Win Rate")

plt.show()

positivly correlated , club power increases , win rate also increases

In [ ]:
#8.pie charts for some category cols
cols_to_plot = ['position', 'foot']
colors = ['hotpink', 'purple', 'yellow', '#9933ff', '#ffb3d9']
# Create one figure with 3 subplots (1 row, 3 columns)
fig, axes = plt.subplots(1, 2, figsize=(18, 6))
# Loop through columns and axes together
for ax, col in zip(axes, cols_to_plot):

    counts = data[col].value_counts()

    ax.pie(counts,
           labels=counts.index,
           autopct='%1.1f%%',
           startangle=90,
           pctdistance=0.85,
           colors=colors,
           textprops={'fontsize': 11})

    ax.set_title(f'Distribution of {col.capitalize()}', fontsize=14, fontweight='bold')
    ax.axis('equal')

# Adjust layout
plt.tight_layout()
plt.show()

most of our players positions are defenders , then midfield. And most of them play with their right foot

In [ ]:
#9.Home vs Away Performance
home_goals = data.groupby('home_club_name')['home_club_goals'].mean()
away_goals = data.groupby('away_club_name')['away_club_goals'].mean()

plt.figure(figsize=(10,5))

plt.plot(home_goals.values[:10], label='Home Goals', marker='o')
plt.plot(away_goals.values[:10], label='Away Goals', marker='o')

plt.title("Home vs Away Performance")
plt.legend()

plt.show()

Home Performance is slightly bigger than Away Performance

In [ ]:
#10.subplots > scatter plots about player performance
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
metrics = [
    ('player_efficiency', 'Player Efficiency'),
    ('attack_contribution', 'Attack Contribution'),
    ('goals', 'Goals'),
    ('assists', 'Assists')
]

for ax, (col, label) in zip(axes.flat, metrics):
    corr = data['market_value_in_eur'].corr(data[col])
    sns.scatterplot(data=data, x=col, y='market_value_in_eur',
                    alpha=0.3, ax=ax, edgecolor='w')
    sns.regplot(data=data, x=col, y='market_value_in_eur',
                scatter=False, color='red', ax=ax)
    ax.set_title(f'{label} vs. Market Value\n(Correlation: {corr:.3f})', fontsize=11)
    ax.set_xlabel(label)
    ax.set_ylabel('Market Value (EUR)')

plt.suptitle('Player Performance vs. Market Value', fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

player performance is positevly correlated with their attack Contribution

PREPROCESSING

feauture engineering for models

In [ ]:
data["total_minutes_player"] = (
    data.groupby("player_name")["minutes_played"].transform("sum")
)
data["total_impact_score"] = (
    data.groupby("player_name")["Player_Impact_Score"].transform("sum")
)
data["avg_player_rating"] = (
    data.groupby("player_name")["player_rating"].transform("mean")
)
data["avg_influence"] = (
    data.groupby("player_name")["influence"].transform("mean")
)
data["total_discipline"] = (
    data.groupby("player_name")["discipline"].transform("sum")
)
data["total_goals_player"] = (
    data.groupby("player_name")["goals"].transform("sum")
)
data["total_assists_player"] = (
    data.groupby("player_name")["assists"].transform("sum")
)
data["avg_player_efficiency"] = (
    data.groupby("player_name")["player_efficiency"].transform("mean")
)

data["avg_player_attack_contribution"] = (
    data.groupby("player_name")["attack_contribution"].transform("mean")
)


In [ ]:
data_analysis =data.copy()

In [ ]:
data_analysis.columns

In [ ]:
data_analysis.to_csv('Full_data_for_analysis.csv')

In [ ]:
# 1. OUTLIER TREATMENT - LOG TRANSFORMATION
print("\n" + "="*50)
print("OUTLIER TREATMENT - LOG TRANSFORMATION")
print("="*50)
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
# Before
sns.histplot(data['market_value_in_eur'], bins=50, kde=True, ax=axes[0], color='red')
axes[0].set_title('Market Value - BEFORE Log Transform')
# Apply Log
data['market_value_log'] = np.log1p(data['market_value_in_eur'])

# After
sns.histplot(data['market_value_log'], bins=50, kde=True, ax=axes[1], color='green')
axes[1].set_title('Market Value - AFTER Log Transform')

plt.tight_layout()
plt.savefig('market_value_log_transform.png', dpi=150)
plt.show()

print(f"Before - Skewness: {data['market_value_in_eur'].skew():.2f}")
print(f"After  - Skewness: {data['market_value_log'].skew():.2f}")


In [ ]:
# # 2. ENCODE CATEGORICAL FEATURES
# print("\n" + "="*50)
# print("ENCODING CATEGORICAL FEATURES")
# print("="*50)

# cat_cols = data.select_dtypes(include=['object']).columns.tolist()
# print(f"Categorical columns: {cat_cols}")

# labels = {}

# for col in cat_cols:
#     le = LabelEncoder()
#     data[col] = le.fit_transform(data[col].astype(str))
#     print(f"   Encoded {col}: {len(le.classes_)} categories")
    
#     labels[col] = le

In [ ]:
# with open("labels_encoding.pkl", "wb") as f:
#     pickle.dump(labels, f)

In [ ]:
cat_cols = data.select_dtypes(include=["object"]).columns.tolist()
labels = {}
for col in cat_cols:
    le = LabelEncoder().fit(data[col].astype(str))
    labels[col] = le
    data[col] = le.transform(data[col].astype(str))


In [ ]:
cat_cols = data.select_dtypes(include=['object']).columns.tolist()
print(f"Categorical columns: {cat_cols}")


for col in cat_cols:
    le = labels[col]
    data[col] = le.transform(data[col].astype(str))
    print(f"   Encoded {col}: {len(le.classes_)} categories")
    

In [ ]:
# # 4. FEATURE SCALING
# print("\n" + "="*50)
# print("FEATURE SCALING")
# print("="*50)

# num_cols = data.select_dtypes(include=['int64', 'float64']).columns

# exclude_cols = ['appearance_id', 'player_name', 'position', 'sub_position', 
#                 'foot', 'club_name', 'opponent_name', 
#                 'home_club_name', 'away_club_name' , 'market_value_in_eur']

# num_cols = [col for col in num_cols if col not in exclude_cols]

# scaler = StandardScaler()
# data[num_cols] = scaler.fit_transform(data[num_cols])

# print(f"Scaled {len(num_cols)} features")

In [ ]:
# with open("scaler.pkl", "wb") as f:
#     pickle.dump(scaler, f)

In [ ]:

scaler = StandardScaler()

In [ ]:
# # 4. FEATURE SCALING
# print("\n" + "="*50)
# print("FEATURE SCALING")
# print("="*50)

# num_cols = data.select_dtypes(include=['int64', 'float64']).columns

# exclude_cols = ['appearance_id', 'player_name', 'position', 'sub_position', 
#                 'foot', 'club_name', 'opponent_name', 
#                 'home_club_name', 'away_club_name' , 'market_value_in_eur']

# num_cols = [col for col in num_cols if col not in exclude_cols]

# data[num_cols] = scaler.fit_transform(data[num_cols])

# print(f"Scaled {len(num_cols)} features")

In [ ]:
print("PREPROCESSING COMPLETE")
print(f"Final shape: {data.shape}")
print("="*50)

In [ ]:
#3.Correlation Heatmap
corr = data.select_dtypes(include='number').corr()

In [ ]:
plt.figure(figsize=(20,10))
sns.heatmap(corr , fmt='.1f' , annot=True , cmap='Blues')
plt.show()

In [ ]:
data_model = data.copy()

In [ ]:
data_model.columns

In [ ]:
data_model = data_model[['age_player' , 'total_goals_player' , 'total_assists_player' , 'total_minutes_player' ,
                        'avg_player_efficiency' ,'total_impact_score' ,'avg_player_rating',
                        'avg_player_attack_contribution' ,'avg_influence' ,
                        'market_value_in_eur','highest_market_value_in_eur' , 'position' ,
                        'sub_position','foot','total_discipline' , 'club_name'  
                        ]]

In [ ]:
data_model = data_model.drop_duplicates()

In [ ]:
x = data_model.drop(columns=["market_value_in_eur"])
y = data_model["market_value_in_eur"]
n_features = x.shape[-1]

In [ ]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import cross_val_score

In [ ]:
fitness_cache = {}

def repair(individual, min_features=12):
    while sum(individual) < min_features:
        i = random.randint(0, len(individual) - 1)
        individual[i] = 1
    return individual


def create_individual():
    individual = [random.randint(0, 1) for _ in range(n_features)]
    return repair(individual, min_features=12)


def fitness(individual , x ,y):
    key = tuple(individual)

    # 🔥 CACHE CHECK
    if key in fitness_cache:
        return fitness_cache[key]

    selected = [i for i in range(n_features) if individual[i] == 1]

    if len(selected) == 0:
        return 0

    model = RandomForestRegressor(
        n_estimators=50,
        max_depth=10,
        min_samples_leaf=2,
        random_state=42
    )

    score = cross_val_score(
        model,
        x.iloc[:, selected],
        y,
        cv=3
    ).mean()

    fitness_cache[key] = score
    return score


def select_parent(population , x , y):
    tournament = random.sample(population, 3)
    return max(tournament, key=lambda ind: fitness(ind, x, y))


def crossover(parent1, parent2):
    point = random.randint(1, n_features - 1)
    return parent1[:point] + parent2[point:]



def mutate(individual):
    individual = individual.copy()

    for i in range(len(individual)):
        if random.random() < 0.05:
            individual[i] = 1 - individual[i]

    # ensure at least 5 features selected
    return repair(individual, min_features=12)


In [ ]:
def GA_Feature_Selection(x ,y):

    population_size = 40
    generations = 15

    population = [create_individual() for _ in range(population_size)]

    best_overall = None
    best_score = -float("inf")

    for generation in range(generations):

        population = sorted(population, key=lambda ind: fitness(ind, x, y), reverse=True)

        best = population[0]
        best_fitness = fitness(best ,x ,y)

        if best_fitness > best_score:
            best_overall = best
            best_score = best_fitness

        print(f"Generation {generation + 1}")
        print(f"Best Fitness: {best_fitness}")
        print("------")

        new_population = []

        # 🔥 ELITISM (keep best)
        new_population.append(best)

        for _ in range(population_size - 1):
            parent1 = select_parent(population ,x ,y)
            parent2 = select_parent(population ,x ,y)

            child = crossover(parent1, parent2)
            child = mutate(child)

            new_population.append(child)

        population = new_population

    # final selected features
    features = [
        x.columns[i]
        for i in range(len(best_overall))
        if best_overall[i] == 1
    ]

    print("\nFinal Solution:")
    print(f"Best Chromosome: {best_overall}")
    print(f"Selected Features: {features}")
    print(f"Best Score: {best_score}")

    return best_overall , features, best_score

In [ ]:
# best_overall , features, best_score = GA_Feature_Selection(x, y)

In [ ]:
import pickle

# results = {
#     "best_overall": best_overall,
#     "features": features,
#     "best_score": best_score
# }

# with open("ga_feature_selection.pkl", "wb") as f:
#     pickle.dump(results, f)

In [ ]:
# The supplied GA implementation is retained above as an optional exercise.
# Use the dashboard's fixed feature set for a reproducible, bounded default run.
features = ['age_player','total_goals_player','total_assists_player','avg_player_efficiency','total_impact_score','avg_player_rating','avg_player_attack_contribution','avg_influence','highest_market_value_in_eur','position','sub_position','foot']
best_overall = [int(col in features) for col in x.columns]
best_score = float('nan')  # No GA validation score is claimed for this default selection.


In [ ]:
final_data_model = data_model[features]
final_data_model

In [ ]:
from sklearn.ensemble import RandomForestRegressor
import matplotlib.pyplot as plt
import pandas as pd
from sklearn.model_selection import train_test_split 

model = RandomForestRegressor(
    n_estimators=100,
    random_state=42
)
X_train,X_test,y_train,y_test=train_test_split(final_data_model,y,shuffle=True,test_size=0.2,random_state=1)

model.fit(X_train, y_train)

In [ ]:
with open("random_forest.pkl", "wb") as f:
    pickle.dump(model, f)

In [ ]:
from sklearn.ensemble import RandomForestRegressor
model = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=2)
model.fit(final_data_model, y)


In [ ]:
importances = model.feature_importances_

features = final_data_model.columns

feature_importance_df = pd.DataFrame({
    "feature": features,
    "importance": importances
})

feature_importance_df = feature_importance_df.sort_values(
    by="importance",
    ascending=False
)

In [ ]:
feature_importance_df

In [ ]:
plt.figure(figsize=(10,6))

plt.barh(
    feature_importance_df["feature"],
    feature_importance_df["importance"]
)

plt.gca().invert_yaxis()  # عشان الأعلى يبقى فوق
plt.title("Feature Importance (Random Forest)")
plt.xlabel("Importance")

plt.show()

In [ ]:
print(data_model.shape)
data_model.head()

In [ ]:
print(final_data_model.shape)
final_data_model.head()

In [ ]:
y.head()